# Inspect images and the descriptions the model wrote for them

Joins an eval run's `raw_responses.jsonl` back to the images and ground truth, so a
number in a table can be traced to the picture that produced it.

Deliberately **no `ipywidgets`** — it is not installed in the `waste_vlm` env, and a
notebook that needs a widget stack is a notebook that stops working on the next
machine. Everything here is a function call you re-run with a different argument.

Two joins are available and they answer different questions:

| source | gives you | note |
|---|---|---|
| `raw_responses.jsonl` | turn-1 description, turn-2 commit, parsed labels | one row per image, in test order |
| `binary_auc.json` | the Yes/No margin and the fitted threshold | separate run, joined by position |

The margin join is **positional**, so the loader below refuses it unless the two runs
have the same length — a silent off-by-one would put every margin on the wrong image
and still look plausible.

In [ ]:
import json, os, pathlib, sys, textwrap

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT))
os.environ.setdefault("WASTE_DATA_ROOT", "/leonardo_scratch/large/userexternal/adiecidu/waste_vlm/data")

EVAL = pathlib.Path("/leonardo_scratch/large/userexternal/adiecidu/waste_vlm/results/vlm_eval")

# --- pick a run -------------------------------------------------------------
ARM      = "stage3_n2d"     # stage3_n1 | stage3_n2 | stage3_n2d | finetune | finetune_next
DATASET  = "aw_m2"          # aw_m2 | aw_m4 | dw_paper10
STYLE    = "open_cot"       # open_cot (has descriptions) | closed_vocab

RUN = EVAL / f"vlm_cradiov4-so_r768ps2_{ARM}_{DATASET}_{STYLE}"
print(RUN, "exists:", RUN.exists())

In [ ]:
def load_run(run: pathlib.Path, dataset: str, with_margins: bool = True):
    """Rows of {image_path, gt, turn1, turn2, parsed, margin?}, in test order."""
    from src.vlm_eval import WASTE_DATA_ROOT, DATASETS

    if dataset in ("aw_m2", "aw_m4"):
        from src.datasets import load_aerialwaste_mcml
        cats, samples = load_aerialwaste_mcml(
            str(WASTE_DATA_ROOT / "aerialwaste"), split="test",
            version="m2" if dataset == "aw_m2" else "m4")
    else:
        from src.datasets import load_dronewaste_multilabel
        cats, samples = load_dronewaste_multilabel(
            str(WASTE_DATA_ROOT / "dronewaste"),
            categories_filter=DATASETS[dataset]["cats"])
    by_id = {s.image_id: s for s in samples}

    rows = []
    for line in (run / "raw_responses.jsonl").open():
        if not line.strip():
            continue
        r = json.loads(line)
        s = by_id.get(str(r["image_id"]))
        rows.append({
            "image_id": r["image_id"],
            "image_path": str(s.image_path) if s else None,
            "gt": r["gt"],
            "turn1": r.get("raw_turn1") or "",
            "turn2": r.get("raw") or "",
            "parsed": r.get("parsed") or [],
        })

    thr = None
    if with_margins:
        arm = run.name.split(f"_{dataset}_")[0].replace("vlm_cradiov4-so_r768ps2_", "")
        f = EVAL / f"binauc_cradiov4-so_r768ps2_{arm}_{dataset}_fit" / "binary_auc.json"
        if f.exists():
            d = json.loads(f.read_text())
            # Positional join: only safe when neither run skipped an image.
            if len(d["scores"]) == len(rows):
                for row, m in zip(rows, d["scores"]):
                    row["margin"] = m
                thr = d["fit_on_train"]["threshold"]
            else:
                print(f"[skip margins] {len(d['scores'])} scores vs {len(rows)} responses "
                      f"-- positional join would misalign every row")
    return rows, cats, thr


rows, CATS, THR = load_run(RUN, DATASET)
print(f"{len(rows)} rows | {sum(bool(r['gt']) for r in rows)} positive | "
      f"threshold {THR if THR is None else round(THR, 3)} | categories {CATS}")

In [ ]:
WASTE_TERMS = ("pile", "debris", "waste", "rubble", "dump", "litter", "trash", "garbage")


def mentions_waste(row) -> bool:
    return any(t in row["turn1"].lower() for t in WASTE_TERMS)


def gate_fires(row) -> bool | None:
    if THR is None or "margin" not in row:
        return None
    return row["margin"] >= THR


def summarise(rows):
    pos = [r for r in rows if r["gt"]]
    neg = [r for r in rows if not r["gt"]]
    print(f"describes waste:      gt+ {np.mean([mentions_waste(r) for r in pos]):.1%}"
          f"   gt- {np.mean([mentions_waste(r) for r in neg]):.1%}")
    print(f"names any category:   gt+ {np.mean([bool(r['parsed']) for r in pos]):.1%}"
          f"   gt- {np.mean([bool(r['parsed']) for r in neg]):.1%}")
    if THR is not None and "margin" in rows[0]:
        print(f"gate fires:           gt+ {np.mean([gate_fires(r) for r in pos]):.1%}"
              f"   gt- {np.mean([gate_fires(r) for r in neg]):.1%}")
    print(f"mean description:     {np.mean([len(r['turn1'].split()) for r in rows]):.0f} words, "
          f"{len({r['turn1'] for r in rows})}/{len(rows)} distinct")


summarise(rows)

## Look at one image

`show(i)` by position, or `show_id("35")` by the id printed in the tables.

In [ ]:
def show(i, rows=None, figsize=(11, 4.4)):
    rows = rows if rows is not None else globals()["rows"]
    r = rows[i] if isinstance(i, int) else i
    fig, (ax_i, ax_t) = plt.subplots(1, 2, figsize=figsize,
                                     gridspec_kw={"width_ratios": [1, 1.3]})
    ax_i.imshow(Image.open(r["image_path"]).convert("RGB"))
    ax_i.set_xticks([]); ax_i.set_yticks([])
    ax_i.set_title(f"image {r['image_id']}", fontsize=9)
    ax_t.axis("off")

    g = gate_fires(r)
    head = f"ground truth: {', '.join(r['gt']) or 'no waste'}"
    if "margin" in r:
        head += f"\nmargin {r['margin']:+.2f}"
        if THR is not None:
            head += f" vs threshold {THR:+.2f}  ->  gate says {'WASTE' if g else 'clean'}"
    body = (f"{head}\n\n"
            f"description:\n{textwrap.fill(r['turn1'] or '(empty)', 62)}\n\n"
            f"commit: {r['turn2'] or '(empty)'}\n"
            f"parsed: {r['parsed'] or '[]'}")
    ax_t.text(0, 1, body, va="top", ha="left", fontsize=8.5, linespacing=1.5,
              family="DejaVu Sans")
    plt.tight_layout(); plt.show()


def show_id(image_id, rows=None):
    rows = rows if rows is not None else globals()["rows"]
    hit = [r for r in rows if str(r["image_id"]) == str(image_id)]
    if not hit:
        raise KeyError(f"image {image_id} not in this run")
    show(hit[0], rows)


show(next(i for i, r in enumerate(rows) if r["gt"] and mentions_waste(r)))

## The four cases worth reading

The aggregate numbers hide which *kind* of mistake is being made. These four buckets
are the ones that drove the conclusions in `EXPERIMENTS.md`:

- **detected and described** — the working path
- **detected but not named** — the naming gap (`n_gate_pos_parser_empty` in the eval)
- **missed** — gate says clean on a real dump
- **false mention** — waste described on a clean tile, the cost of `n2d`'s descriptions

In [ ]:
BUCKETS = {
    "detected and described": lambda r: r["gt"] and gate_fires(r) and mentions_waste(r),
    "detected but not named": lambda r: r["gt"] and gate_fires(r) and not r["parsed"],
    "missed":                 lambda r: r["gt"] and gate_fires(r) is False,
    "false mention":          lambda r: not r["gt"] and mentions_waste(r),
}

buckets = {k: [r for r in rows if f(r)] for k, f in BUCKETS.items()}
for k, v in buckets.items():
    print(f"{k:26s} {len(v):5d}")

In [ ]:
# Change the bucket name / index to walk through examples.
BUCKET, N = "detected but not named", 3

for r in buckets[BUCKET][:N]:
    show(r)

## Compare two arms on the same image

This is what turned the description finding around: `n1` and `n2d` score similarly on
detection and write completely different text.

In [ ]:
def compare(image_id, arms=("stage3_n1", "stage3_n2d", "stage3_s3b"), dataset=DATASET):
    for arm in arms:
        run = EVAL / f"vlm_cradiov4-so_r768ps2_{arm}_{dataset}_open_cot"
        if not run.exists():
            print(f"-- {arm}: no run"); continue
        rs, _, _ = load_run(run, dataset, with_margins=False)
        hit = [r for r in rs if str(r["image_id"]) == str(image_id)]
        if not hit:
            print(f"-- {arm}: image not in run"); continue
        r = hit[0]
        print(f"-- {arm}   commit={r['turn2']!r} parsed={r['parsed']}")
        print(textwrap.indent(textwrap.fill(r["turn1"] or "(empty)", 88), "     "))
        print()


compare(rows[next(i for i, r in enumerate(rows) if r["gt"] and mentions_waste(r))]["image_id"])